Fifth42 exact GPU packet benchmark

Select **Runtime → Change runtime type → L4 GPU** for the first test. All sources and inputs download from a fixed GitHub commit; no file uploads are needed. This is a bounded benchmark with no new scientific coverage. Keep your Colab Pro notebook open during execution.


In [ ]:
import os, platform, subprocess
from pathlib import Path
print("Python:", platform.python_version())
print("CPU affinity count:", len(os.sched_getaffinity(0)))
print(next(line for line in Path("/proc/meminfo").read_text().splitlines() if line.startswith("MemTotal:")))
for command in [["nvidia-smi"], ["nvcc", "--version"], ["g++", "--version"]]:
    result = subprocess.run(command, text=True, capture_output=True, timeout=10)
    print(result.stdout or result.stderr)


Download and verify the complete benchmark bundle from GitHub. Each transport chunk, the reconstructed archive, and all extracted files are checked before execution.


In [ ]:
import base64, concurrent.futures, hashlib, io, json, tarfile, tempfile, urllib.request
ASSET_BASE = 'https://raw.githubusercontent.com/paulnord/fifth42/4668a27951f2b9d7d533896d1aa32684808341d4/colab/20261112'
EXPECTED_MANIFEST_SHA256 = '57d6965885da6c397f5cc1265fe71d15e7d1bc9c95434a3e166343f14f03bcae'
with urllib.request.urlopen(ASSET_BASE + "/transport/manifest.json", timeout=30) as response:
    manifest_raw = response.read()
assert hashlib.sha256(manifest_raw).hexdigest() == EXPECTED_MANIFEST_SHA256, "Manifest checksum mismatch"
manifest = json.loads(manifest_raw)
def fetch_part(entry):
    assert entry["name"].startswith("part-") and "/" not in entry["name"]
    with urllib.request.urlopen(ASSET_BASE + "/transport/" + entry["name"], timeout=30) as response:
        raw = response.read()
    assert len(raw) == entry["bytes"] and hashlib.sha256(raw).hexdigest() == entry["sha256"], "Chunk checksum mismatch"
    return raw
with concurrent.futures.ThreadPoolExecutor(max_workers=4) as pool:
    encoded_parts = list(pool.map(fetch_part, manifest["chunks"]))
archive_data = base64.b64decode(b"".join(encoded_parts), validate=True)
assert len(archive_data) == manifest["archive_bytes"]
assert hashlib.sha256(archive_data).hexdigest() == manifest["archive_sha256"], "Archive checksum mismatch"
FIFTH42_DIR = Path(tempfile.mkdtemp(prefix="fifth42-tests-", dir="/content"))
seen = set()
with tarfile.open(fileobj=io.BytesIO(archive_data), mode="r:gz") as archive:
    for member in archive.getmembers():
        rel = Path(member.name)
        assert member.isfile() and not rel.is_absolute() and ".." not in rel.parts and member.name not in seen
        seen.add(member.name)
        target = FIFTH42_DIR / rel
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_bytes(archive.extractfile(member).read())
for line in (FIFTH42_DIR / "SHA256SUMS").read_text().splitlines():
    digest, name = line.split("  ", 1)
    assert hashlib.sha256((FIFTH42_DIR / name).read_bytes()).hexdigest() == digest, name
print("Verified GitHub source/data:", FIFTH42_DIR)
print("Asset commit:", ASSET_BASE)


Build for this runtime’s actual GPU using `-arch=native`. If GMP headers or a C++ compiler are missing, install `libgmp-dev`/`g++` in this Colab runtime and rerun this cell. Preserve compiler compatibility errors for review.


In [ ]:
subprocess.run(["python3", "build.py", "--gpu"], cwd=FIFTH42_DIR, check=True, timeout=180)


Verify all exact results and benchmark. The realistic packet stream is already full enough that coalescing leaves its 23 requests byte-identical, so that group gets a baseline measurement without a fabricated speedup ratio. The tiny group compares 16 requests with one combined request.


In [ ]:
subprocess.run(["python3", "colab_bench.py", "--backend", "gpu", "--repeats", "7", "--max-seconds", "180", "--verify-gmp"], cwd=FIFTH42_DIR, check=True, timeout=240)


In [ ]:
report = json.loads((FIFTH42_DIR / "results/benchmark-gpu.json").read_text())
print(json.dumps({key: report[key] for key in ["status", "hardware", "sizes", "summary", "elapsed_seconds"]}, indent=2))


Download results before disconnecting. Also record the compute-unit rate and starting/ending balance from Colab’s UI if available; those account values are not inferred by this notebook.


In [ ]:
import zipfile
result_zip = FIFTH42_DIR / "Fifth42_Colab_Results.zip"
with zipfile.ZipFile(result_zip, "w", zipfile.ZIP_DEFLATED) as archive:
    for path in sorted((FIFTH42_DIR / "results").glob("*.json")):
        archive.write(path, path.relative_to(FIFTH42_DIR))
    for name in ["SHA256SUMS", "data/export.json"]:
        archive.write(FIFTH42_DIR / name, name)
files.download(str(result_zip))
